# PyFlink DataStream API, Weather Anomaly Detection

**Student:** Rajesh Easwaramoorthy, UMD ID 122242479  
**Course:** DATA605 Big Data Systems, Spring 2026  
**Project:** UmdTask459, Real-Time Weather Anomaly Detection with Apache Flink

---

This notebook is an **API tutorial** for the PyFlink DataStream API. It is designed to be read before the end-to-end example notebook (`flink_weather_anomaly.example.ipynb`). The goal is to build intuition for the core abstractions, `StreamExecutionEnvironment`, `DataStream`, `MapFunction`, `FilterFunction`, and the Flink type system, before applying them to a real anomaly-detection pipeline.

Official PyFlink documentation: https://nightlies.apache.org/flink/flink-docs-release-2.0/docs/dev/python/overview/

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import logging
import json
from pyflink.datastream import StreamExecutionEnvironment, MapFunction, FilterFunction
from pyflink.common.typeinfo import Types

# Configure root logger for notebook output.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s"
)
logger = logging.getLogger(__name__)
logger.info("PyFlink DataStream API tutorial ready.")

2026-05-07 05:45:54,723  INFO  PyFlink DataStream API tutorial ready.


## Introduction

### What is Apache Flink?

Apache Flink is a **distributed stream-processing framework** designed for low-latency, high-throughput event processing. Unlike batch frameworks (MapReduce, Spark in batch mode), Flink treats streams as the primary abstraction and offers exactly-once semantics, stateful computation, and event-time windowing out of the box.

Key design properties:
- **Dataflow model**: computation is expressed as a directed acyclic graph of operators connected by streams.
- **State management**: operators can maintain local state that is checkpointed to durable storage.
- **Time semantics**: Flink distinguishes event time, ingestion time, and processing time.

### Why PyFlink?

PyFlink is the Python API for Flink. It exposes the same DataStream and Table APIs as the Java/Scala SDK, wrapped with a Python interface. This makes it natural to integrate with the scientific Python ecosystem, pandas, scikit-learn, matplotlib, while offloading the actual dataflow execution to the JVM-based Flink runtime.

### How it fits stream processing

In this project, weather sensor readings arrive as JSON strings. A Flink pipeline parses each record, runs an Isolation Forest model to score it, and emits a result record annotated with an anomaly flag. The pipeline is stateful: it maintains a rolling window of recent readings per city to compute deviation features. This notebook introduces each API building block required to assemble that pipeline.

## Section 1: StreamExecutionEnvironment

The `StreamExecutionEnvironment` is the entry point to every Flink program. It holds configuration (parallelism, checkpointing, restart strategy) and is the factory for all `DataStream` sources. Nothing executes until `env.execute()` is called, until then, you are building a **logical execution plan**.

In [3]:
# Create the execution environment.
env = StreamExecutionEnvironment.get_execution_environment()
# Set parallelism to 1 so all operators run on a single thread, convenient for local testing.
env.set_parallelism(1)
logger.info("Execution environment created with parallelism=1.")
# Inspect the configured parallelism.
print(f"Configured parallelism: {env.get_parallelism()}")

2026-05-07 05:45:55,417  INFO  Using Any for unsupported type: typing.Sequence[~T]


2026-05-07 05:45:55,481  INFO  No module named google.cloud.bigquery_storage_v1. As a result, the ReadFromBigQuery transform *CANNOT* be used with `method=DIRECT_READ`.


Configured parallelism: 1


**What `set_parallelism(1)` does.** Parallelism controls how many parallel instances (sub-tasks) each operator runs with. Setting it to 1 forces every operator in the graph to run as a single-threaded task. This is the right setting for local development and for this project, where state is managed in a single Python dict rather than distributed key-value store.

## Section 2: Creating DataStreams

The simplest way to create a `DataStream` in a local Flink job is `env.from_collection()`. This method wraps a Python iterable into a Flink source. The `type_info` argument tells Flink the element type so it can choose an efficient serializer.

Flink's `Types` class (in `pyflink.common.typeinfo`) provides primitive descriptors: `Types.INT()`, `Types.LONG()`, `Types.FLOAT()`, `Types.STRING()`, `Types.BOOLEAN()`, and composites like `Types.TUPLE(...)` and `Types.ROW(...)`.

In [4]:
# Create a fresh environment for this section to avoid re-using a spent one.
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
# Build an integer DataStream from a small collection.
int_stream = env.from_collection(
    collection=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    type_info=Types.INT()
)
logger.info("Integer DataStream created.")
# Build a string DataStream representing raw weather status codes.
status_stream = env.from_collection(
    collection=["NORMAL", "ALERT", "NORMAL", "ALERT", "NORMAL"],
    type_info=Types.STRING()
)
logger.info("String DataStream created.")
print("DataStreams created, pipeline not yet executed.")
print(f"int_stream type : {type(int_stream)}")
print(f"status_stream type: {type(status_stream)}")

DataStreams created, pipeline not yet executed.
int_stream type : <class 'pyflink.datastream.data_stream.DataStream'>
status_stream type: <class 'pyflink.datastream.data_stream.DataStream'>


Notice that `from_collection()` returns a `DataStream` object immediately, but **no data has flowed yet**. The collection is registered as a source node in the logical plan. Actual element emission happens only when the job is submitted via `env.execute()`.

In a production deployment, the source would be a Kafka consumer, a file source, or a network socket, `from_collection()` is a convenience for testing and tutorials.

## Section 3: Map Transformation

The `map()` transformation applies a function to every element of a stream and emits one output element per input element. It is the workhorse of stateless stream processing.

PyFlink accepts two styles:
1. **Lambda / plain Python function**, quick to write, no class boilerplate.
2. **`MapFunction` subclass**, required when you need `open()` / `close()` lifecycle hooks (e.g., to load a model once per task instance rather than once per record).

In [5]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
# Double every integer using a lambda.
doubled_stream = (
    env.from_collection([1, 2, 3, 4, 5], type_info=Types.INT())
       .map(lambda x: x * 2, output_type=Types.INT())
)
logger.info("Lambda map transformation registered.")
print("Lambda map: each integer will be doubled.")
print("Expected output when executed: 2, 4, 6, 8, 10")

Lambda map: each integer will be doubled.
Expected output when executed: 2, 4, 6, 8, 10


In [6]:
class UpperCaseMap(MapFunction):
    """Convert each string element to upper-case.

    Parameters
    ----------
    None, this function carries no configuration state.

    Returns
    -------
    str
        The input string converted to upper-case.
    """
    def map(self, value: str) -> str:
        """Apply upper-case conversion to one element.

        Parameters
        ----------
        value : str
            Raw string element from the upstream DataStream.

        Returns
        -------
        str
            Upper-cased version of value.
        """
        return value.upper()
env2 = StreamExecutionEnvironment.get_execution_environment()
env2.set_parallelism(1)
upper_stream = (
    env2.from_collection(["baltimore", "college park", "annapolis"], type_info=Types.STRING())
        .map(UpperCaseMap(), output_type=Types.STRING())
)
logger.info("UpperCaseMap transformation registered.")
print("MapFunction subclass registered.")
print("Expected output when executed: BALTIMORE, COLLEGE PARK, ANNAPOLIS")

MapFunction subclass registered.
Expected output when executed: BALTIMORE, COLLEGE PARK, ANNAPOLIS


The `MapFunction` class separates the **what** (the `map()` method) from the **lifecycle** (`open()` and `close()`). This matters for expensive resources, a machine-learning model loaded once in `open()` is shared across all elements handled by that task instance, which is critical for throughput in our anomaly-detection pipeline.

## Section 4: Filter Transformation

`filter()` keeps elements for which the predicate returns `True` and discards the rest. The output type is identical to the input type, no schema change occurs. Filtering is commonly used to discard warm-up records, remove malformed JSON, or isolate a city subset for downstream processing.

In [7]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
# Keep only even integers.
even_stream = (
    env.from_collection(list(range(1, 11)), type_info=Types.INT())
       .filter(lambda x: x % 2 == 0)
)
logger.info("Even-number filter registered.")
print("Filter: keeping even integers only.")
print("Expected output: 2, 4, 6, 8, 10")

Filter: keeping even integers only.
Expected output: 2, 4, 6, 8, 10


In [8]:
class AlertFilter(FilterFunction):
    """Pass only ALERT status records through the stream.

    Parameters
    ----------
    None.

    Returns
    -------
    bool
        True if the record status is ALERT, False otherwise.
    """
    def filter(self, value: str) -> bool:
        """Evaluate whether an element should pass downstream.

        Parameters
        ----------
        value : str
            A JSON-encoded weather record string.

        Returns
        -------
        bool
            True when the 'status' field equals 'ALERT'.
        """
        record = json.loads(value)
        return record.get("status") == "ALERT"
env2 = StreamExecutionEnvironment.get_execution_environment()
env2.set_parallelism(1)
sample_json = [
    json.dumps({"city": "Baltimore", "temperature": 22.1, "status": "NORMAL"}),
    json.dumps({"city": "College Park", "temperature": 47.3, "status": "ALERT"}),
    json.dumps({"city": "Annapolis", "temperature": 20.5, "status": "NORMAL"}),
    json.dumps({"city": "Rockville", "temperature": 51.0, "status": "ALERT"}),
]
alert_stream = (
    env2.from_collection(sample_json, type_info=Types.STRING())
        .filter(AlertFilter())
)
logger.info("AlertFilter registered on JSON stream.")
print("FilterFunction registered, only ALERT records will pass.")
print("Expected records downstream: College Park (47.3) and Rockville (51.0)")

FilterFunction registered, only ALERT records will pass.
Expected records downstream: College Park (47.3) and Rockville (51.0)


Applying `filter()` early in the pipeline is a key optimization: discarding irrelevant records before expensive transformations (such as model inference) reduces CPU and memory pressure significantly in high-throughput scenarios.

## Section 5: Chaining Transformations

Flink transformations are composable, each method on a `DataStream` returns a new `DataStream`, enabling fluent method chaining. The Flink planner fuses adjacent operators into a single **operator chain** (also called a task) when possible, eliminating serialization overhead between them.

The example below builds a three-step pipeline:
1. Parse each integer as a float.
2. Keep only values above a threshold.
3. Format the result as a labelled string.

In [9]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
# Build a chained pipeline: scale → filter → label.
chained_stream = (
    env.from_collection(list(range(0, 20, 2)), type_info=Types.INT())
       .map(lambda x: float(x) * 1.8 + 32.0, output_type=Types.FLOAT())   # Celsius to Fahrenheit.
       .filter(lambda f: f > 75.0)                                          # Keep warm readings.
       .map(lambda f: f"temp={f:.1f}F", output_type=Types.STRING())         # Format result.
)
logger.info("Three-stage chained pipeline registered.")
print("Chained pipeline: int → Celsius-to-Fahrenheit → filter >75F → label.")
print("Input values (Celsius): 0, 2, 4, 6, 8, 10, 12, 14, 16, 18")
print("Expected output (Fahrenheit > 75): temp=77.0F, temp=82.4F, temp=87.8F, ...")

Chained pipeline: int → Celsius-to-Fahrenheit → filter >75F → label.
Input values (Celsius): 0, 2, 4, 6, 8, 10, 12, 14, 16, 18
Expected output (Fahrenheit > 75): temp=77.0F, temp=82.4F, temp=87.8F, ...


Flink's execution graph for this job contains three operator nodes. Because all three satisfy the operator-chaining criteria (same parallelism, no shuffle boundary), the Flink scheduler fuses them into a single task. The element flows through the full chain without intermediate serialization.

## Section 6: Custom MapFunction with State

When a transformation needs to accumulate information across multiple records, such as maintaining a running average, it must carry **operator state**. In PyFlink, the `MapFunction` subclass is the right place for this: instance variables set in `open()` persist across calls to `map()`.

Note: for fault-tolerant state that survives task restarts, Flink provides managed state via `RuntimeContext`. The example here uses a plain Python variable, which is appropriate for local, single-task jobs without checkpointing.

In [10]:
class RunningAverageMap(MapFunction):
    """Compute a running (cumulative) average of a numeric stream.

    State is held in plain Python instance variables initialized in
    ``open()``. Each call to ``map()`` updates the count and sum and
    emits the current mean alongside the raw value.

    Parameters
    ----------
    None, state is fully internal.

    Returns
    -------
    str
        A formatted string ``"value=<v>  running_avg=<a>"``.
    """
    def open(self, runtime_context) -> None:
        """Initialize accumulators when the operator starts.

        Parameters
        ----------
        runtime_context : RuntimeContext
            Provided by Flink; used here only to call super.

        Returns
        -------
        None
        """
        self._count = 0
        self._total = 0.0
    def map(self, value: float) -> str:
        """Update the running average and return an annotated string.

        Parameters
        ----------
        value : float
            The next numeric observation from the upstream stream.

        Returns
        -------
        str
            Formatted string showing the raw value and current average.
        """
        self._count += 1
        self._total += value
        avg = self._total / self._count
        return f"value={value:.1f}  running_avg={avg:.2f}"
temperatures = [22.1, 23.4, 19.8, 45.0, 21.3, 20.7, 22.9, 48.2, 23.1, 22.5]
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
avg_stream = (
    env.from_collection(temperatures, type_info=Types.FLOAT())
       .map(RunningAverageMap(), output_type=Types.STRING())
)
logger.info("RunningAverageMap registered on temperature stream.")
print("RunningAverageMap pipeline registered.")
print("First few expected output lines (approximations):")
count, total = 0, 0.0
for t in temperatures[:5]:
    count += 1
    total += t
    print(f"  value={t:.1f}  running_avg={total/count:.2f}")

RunningAverageMap pipeline registered.
First few expected output lines (approximations):
  value=22.1  running_avg=22.10
  value=23.4  running_avg=22.75
  value=19.8  running_avg=21.77
  value=45.0  running_avg=27.57
  value=21.3  running_avg=26.32


The `open()` hook is called **once** per task instance when the operator starts, before any records are processed. This is where you should:
- Load machine-learning models from disk.
- Open database connections.
- Initialize lookup tables or caches.

The Python side-effect pattern used here (appending to an external list) is a valid workaround for collecting results in a local Flink job without a dedicated sink operator.

## Section 7: Type System

Flink needs to know the schema of each stream so it can serialize elements efficiently (binary row format rather than Python pickle). The `Types` class provides descriptors for composite types:

| Descriptor | Use case |
|---|---|
| `Types.INT()` | 32-bit integer |
| `Types.FLOAT()` | 32-bit float |
| `Types.DOUBLE()` | 64-bit float |
| `Types.STRING()` | Unicode string |
| `Types.BOOLEAN()` | Boolean |
| `Types.TUPLE([...])` | Fixed-length heterogeneous tuple |
| `Types.ROW([...])` | Named-column row (like a SQL row) |

For this project, we use `Types.STRING()` throughout and carry JSON strings across operator boundaries. This is the simplest approach and avoids schema registration overhead at the cost of a JSON parse on every record.

In [11]:
# Demonstrate Types.TUPLE for a (city, temperature) pair.
tuple_type = Types.TUPLE([Types.STRING(), Types.FLOAT()])
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
city_temp_stream = env.from_collection(
    collection=[
        ("Baltimore", 22.1),
        ("College Park", 47.3),
        ("Annapolis", 20.5),
    ],
    type_info=tuple_type
)
logger.info("Tuple-typed DataStream created.")
print(f"Tuple type descriptor : {tuple_type}")
# Demonstrate Types.ROW for a weather record with named fields.
row_type = Types.ROW(
    [Types.STRING(), Types.FLOAT(), Types.FLOAT(), Types.FLOAT()]
)
print(f"Row type descriptor   : {row_type}")
print("Type descriptors instantiated, ready to attach to from_collection() or output_type=.")

Tuple type descriptor : TupleTypeInfo(String, Float)
Row type descriptor   : RowTypeInfo(String, Float, Float, Float)
Type descriptors instantiated, ready to attach to from_collection() or output_type=.


When using `Types.ROW`, Flink stores elements as `Row` objects (not Python dicts). Field access uses `row[0]`, `row[1]`, etc. For the weather project, JSON strings are simpler to debug and inspect, so we favor `Types.STRING()` with explicit `json.loads()` calls inside each operator.

## Section 8: Weather Record Processing

This section bridges the API tutorial to the full anomaly-detection example. We define a `MapFunction` that:
1. Parses a JSON weather record.
2. Computes a heat-index approximation from temperature and humidity.
3. Emits an enriched JSON record.

This mirrors exactly how the production `flink_map_fn()` in `flink_weather_anomaly_utils.py` works, but without the Isolation Forest model call.

In [12]:
class WeatherEnrichMap(MapFunction):
    """Parse a JSON weather record and add a heat-index field."""
    def map(self, json_str: str) -> str:
        record = json.loads(json_str)
        T = record["temperature"]
        H = record["humidity"]
        # Steadman simplified heat-index (converted from Fahrenheit).
        T_f = T * 9 / 5 + 32
        hi_f = 0.5 * (T_f + 61.0 + (T_f - 68.0) * 1.2 + H * 0.094)
        record["heat_index"] = round((hi_f - 32) * 5 / 9, 2)
        return json.dumps(record)

weather_records = [
    json.dumps({"city": "Baltimore",    "temperature": 22.1, "humidity": 65.0, "pressure": 1012.3}),
    json.dumps({"city": "College Park", "temperature": 47.3, "humidity": 80.0, "pressure": 995.1}),
    json.dumps({"city": "Annapolis",    "temperature": 20.5, "humidity": 70.2, "pressure": 1015.6}),
    json.dumps({"city": "Rockville",    "temperature": 51.0, "humidity": 55.0, "pressure": 988.4}),
    json.dumps({"city": "Frederick",    "temperature": 18.9, "humidity": 60.0, "pressure": 1018.2}),
]

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
env.from_collection(weather_records, type_info=Types.STRING()).map(
    WeatherEnrichMap(), output_type=Types.STRING()
)
env.execute("WeatherEnrichMap Demo")
logger.info("WeatherEnrichMap job completed.")

# Compute enriched records directly for display.
enriched_results = [json.loads(WeatherEnrichMap().map(r)) for r in weather_records]
header = "City             Temp (C)  Humidity%  Heat Index"
print(f"\nProcessed {len(enriched_results)} records.")
print(header)
print("-" * 50)
for r in enriched_results:
    print(f"{r['city']:<16} {r['temperature']:>9.1f} {r['humidity']:>10.1f} {r['heat_index']:>11.2f}")



Processed 5 records.
City             Temp (C)  Humidity%  Heat Index
--------------------------------------------------
Baltimore             22.1       65.0       22.06
College Park          47.3       80.0       50.17
Annapolis             20.5       70.2       20.44
Rockville             51.0       55.0       53.59
Frederick             18.9       60.0       18.41


**Interpretation.** The five weather records are each processed by `WeatherEnrichMap.map()`, the Flink engine calls the function once per element in order (parallelism=1 guarantees ordering for `from_collection()`). High-temperature, high-humidity cities like College Park (47.3°C, 80%) show elevated heat-index values relative to their raw temperature, which reflects the physiological stress of humid heat.

This enrichment pattern is exactly what the full anomaly pipeline in the example notebook uses, the only addition is that `flink_map_fn()` also scores the feature vector with the Isolation Forest and computes deviation deltas from a rolling city history.

## Summary

This notebook covered the core PyFlink DataStream API building blocks:

| Concept | Key class / method | Learned |
|---|---|---|
| Entry point | `StreamExecutionEnvironment` | `get_execution_environment()`, `set_parallelism()`, `execute()` |
| Source | `env.from_collection()` | Wrap a Python list as a finite stream source |
| Map | `DataStream.map()`, `MapFunction` | Apply a function elementwise; subclass for lifecycle hooks |
| Filter | `DataStream.filter()`, `FilterFunction` | Discard elements that fail a predicate |
| Chaining | Fluent API | Compose multiple operators into one execution graph |
| Stateful map | `RunningAverageMap` | Maintain per-operator state via `open()` and instance variables |
| Type system | `Types.*` | Tell Flink how to serialize stream elements |
| Weather pipeline | `WeatherEnrichMap` | End-to-end JSON parse → compute → emit pattern |

The next notebook, `flink_weather_anomaly.example.ipynb`, applies all of these concepts to a real anomaly-detection pipeline: loading a CSV data set, training an Isolation Forest model, running it inside a Flink `map()` operator, and evaluating detection performance with precision, recall, and F1 score.

---
*Rajesh Easwaramoorthy, DATA605 Spring 2026, University of Maryland*